## Sanitização das Bases de Dados da Olist

**Contexto:** A equipe de Engenharia de Dados da Olist extraiu lotes de dados
(`olist_products_dataset.csv` e `olist_orders_dataset.csv`) e identificou inconsistências
que estão travando os relatórios automatizados.

**Escopo deste pipeline:** implementar uma lógica de sanitização completa em ambas as bases de dados percorrendo as seguintes etapas:

* Validação e Tratamento de Dados Ausentes;
* Padronização de Strings e Regex;
* Regras de Negócio;
* Formatação Temporal;
* Relatório de Status Manual.

## Importação dos módulos utilizados

Apenas módulos da biblioteca padrão do Python são utilizados neste pipeline:

In [6]:
import csv
import os
import re
from datetime import datetime

## Funções utilizadas para tratamento dos dados:

As funções que servirão para as etapas de tratamento e sanitização dos dados são declaradas abaixo para fim de encapsulamento do código do notebook

In [7]:
# =========================================================
# Funções de leitura de arquivos de dados
# =========================================================

# Leitura do arquivo de produtos
def ler_arquivo_produtos(caminho_arquivo):
    """
    Le o arquivo de produtos de forma iterativa (linha a linha),
    mantendo as chaves do dicionario iguais aos cabecalhos do CSV.

    Args:
        caminho_arquivo (str): Caminho para o arquivo .csv dos produtos.

    Returns:
        list: Lista de dicionarios, um para cada produto (linha do arquivo).
    """
    produtos = []  # estrutura de dados em memoria (lista de dicionarios)

    # Gerenciador de contexto: garante o fechamento correto do arquivo
    with open(caminho_arquivo, mode="r", encoding="utf-8", newline="") as arquivo:
        leitor = csv.DictReader(arquivo)  # converte cada linha em dicionario

        for linha in leitor:              # varredura linha a linha
            produtos.append(linha)        # acumula o registro em memoria

    return produtos


# Leitura do arquivo de pedidos
def ler_arquivo_pedidos(caminho_arquivo):
    """
    Le o arquivo de pedidos de forma iterativa (linha a linha).

    Args:
        caminho_arquivo (str): Caminho para o arquivo .csv dos pedidos.

    Returns:
        list: Lista de dicionarios, um para cada pedido (linha do arquivo).
    """
    pedidos = []

    with open(caminho_arquivo, mode="r", encoding="utf-8", newline="") as arquivo:
        leitor = csv.DictReader(arquivo)
        for linha in leitor:
            pedidos.append(linha)

    return pedidos


# =========================================================
# Funções de validação e tratamento de dados ausentes
# =========================================================

# Detecção de valores nulos/vazios
def testar_valor_ausente(valor):
    """
    Verifica se um valor extraido do CSV e nulo ou vazio.

    Args:
        valor: Valor bruto da coluna (geralmente str ou None).

    Returns:
        bool: True se o campo estiver nulo/vazio, False caso contrario.
    """
    if valor is None:
        return True
    # Remove espacos e verifica se sobrou algum conteudo
    return str(valor).strip() == ""


# Tratamento de dados ausentes
def padronizar_categoria_ausente(produto):
    """
    Preenche o campo product_category_name com 'sem categoria'
    quando o valor estiver nulo ou vazio.

    Args:
        produto (dict): Registro individual do produto.

    Returns:
        bool: True se houve correcao, False caso contrario.
    """
    if testar_valor_ausente(produto.get("product_category_name")):
        produto["product_category_name"] = "sem categoria"
        return True
    return False

# Cálculo das médias das dimensões físicas dos produtos
COLUNAS_DIMENSOES_FISICAS = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]

def calcular_medias_dimensoes(produtos):
    """
    Calcula a média aritmética de cada dimensão física considerando
    apenas os registros preenchidos (ausentes sao ignorados).
    Para definir a regra de corte, precisamos do valor de referência
    das dimensões. A imputação pela média de cada coluna foi escolhida
    porque, após inspecionar a base de dados, apenas 2 registros possuem
    dimensões físicas ausentes. A imputação pela média não irá impactar a
    média amostral da coluna, já que a proporção de dados ausentes é mínima
    e o seu descarte poderia remover linhas com informações úteis em outras
    colunas.

    Args:
        produtos (list): Lista de dicionarios com os produtos brutos.

    Returns:
        dict: Mapeia cada coluna fisica para sua media calculada.
    """
    medias = {}

    for coluna in COLUNAS_DIMENSOES_FISICAS:
        somatorio = 0.0
        contagem = 0

        for produto in produtos:
            valor = produto.get(coluna)

            # Soma apenas valores numéricos válidos
            if not testar_valor_ausente(valor):
                somatorio += float(valor)   # converte string numerica para float
                contagem += 1

        # Regra de seguranca: se nenhum valor válido, a média assume 0.0
        medias[coluna] = somatorio / contagem if contagem > 0 else 0.0

    return medias


def padronizar_categoria_ausente(produto):
    """
    Preenche o campo product_category_name com 'sem categoria'
    quando o valor estiver nulo ou vazio.

    Args:
        produto (dict): Registro individual do produto.

    Returns:
        bool: True se houve correcao, False caso contrario.
    """
    if testar_valor_ausente(produto.get("product_category_name")):
        produto["product_category_name"] = "sem categoria"
        return True
    return False


def padronizar_dimensoes_ausentes(produto, medias):
    """
    Preenche as dimensões físicas ausentes com a média da coluna.

    Escolha tecnica:
      - Somente 2 registros possuem dimensões ausentes;
      - Imputar pela media preserva o registro e e estatisticamente neutro
        (não altera a media amostral da coluna);
      - Evita o descarte de linhas que possuem informação útil em outras
        colunas (ex.: categoria, descricao, quantidade de fotos).

    Args:
        produto (dict): Registro individual do produto.
        medias (dict): Medias por coluna física.

    Returns:
        int: Quantidade de colunas fisicas corrigidas neste produto.
    """
    correcoes = 0

    for coluna in COLUNAS_DIMENSOES_FISICAS:
        if testar_valor_ausente(produto.get(coluna)):
            # Valor padrao = media da coluna, formatado com 2 casas decimais
            produto[coluna] = f"{medias[coluna]:.2f}"
            correcoes += 1

    return correcoes


def sanitizar_produtos(caminho_arquivo):
    """
    Organiza a lógica de sanitizacao da base de produtos.

    Args:
        caminho_arquivo (str): Caminho do arquivo .csv de produtos.

    Returns:
        tuple: (produtos_sanitizados, total_linhas, categorias_corrigidas,
                dimensoes_corrigidas_por_coluna, medias_utilizadas)
    """
    # Leitura iterativa da base
    produtos = ler_arquivo_produtos(caminho_arquivo)
    total_linhas = len(produtos)

    # Médias de referencia calculadas ANTES da imputação
    medias = calcular_medias_dimensoes(produtos)

    # Aplicação das regras de tratamento
    categorias_corrigidas = 0
    dimensoes_corrigidas = {coluna: 0 for coluna in COLUNAS_DIMENSOES_FISICAS}

    for produto in produtos:
        if padronizar_categoria_ausente(produto):
            categorias_corrigidas += 1

        # Conta valores ausentes por coluna física ANTES da imputacao
        for coluna in COLUNAS_DIMENSOES_FISICAS:
            if testar_valor_ausente(produto.get(coluna)):
                dimensoes_corrigidas[coluna] += 1

        # Preenche as dimensões ausentes (aplica a regra de corte)
        padronizar_dimensoes_ausentes(produto, medias)

    return produtos, total_linhas, categorias_corrigidas, dimensoes_corrigidas, medias


def escrever_produtos_sanitizados(produtos, caminho_saida):
    """
    Grava o dataset sanitizado em um novo arquivo CSV,
    mantendo os mesmos cabeçalhos do arquivo original.

    Args:
        produtos (list): Lista de dicionários já sanitizados.
        caminho_saida (str): Caminho do arquivo de saída.

    Returns:
        int: Quantidade de linhas gravadas.
    """
    if not produtos:
        return 0

    cabecalhos = list(produtos[0].keys())

    with open(caminho_saida, mode="w", encoding="utf-8", newline="") as arquivo:
        escritor = csv.DictWriter(arquivo, fieldnames=cabecalhos)
        escritor.writeheader()   # grava a linha de cabecalho
        for produto in produtos:
            escritor.writerow(produto)  # grava cada registro

    return len(produtos)


# =========================================================
# Funções de padronização de Strings e Regex
# =========================================================

def padronizar_categoria(categoria):
    """
    Padroniza uma string de categoria de produto.

    Etapas:
        1. .strip()  -> remove espacos em branco nas bordas;
        2. .lower()  -> converte tudo para letras minusculas;
        3. regex     -> remove caracteres especiais/pontuação indevidos.

    Args:
        categoria (str): Valor bruto da coluna product_category_name.

    Returns:
        str: Categoria padronizada (ou 'sem categoria' se ausente).
    """
    if testar_valor_ausente(categoria):
        return "sem categoria"

    # Ppadronização básica de strings
    categoria = categoria.strip().lower()

    # Remove caracteres especiais/pontuação (mantém letras, dígitos, _ e espaco)
    categoria = re.sub(r"[^a-z0-9_ ]", "", categoria)

    # Extra: colapsa espacos múltiplos em um único espaço
    categoria = re.sub(r"\s+", " ", categoria).strip()

    return categoria


def aplicar_padronizacao_categorias(produtos):
    """
    Aplica a padronização de strings em todos os produtos.

    Args:
        produtos (list): Lista de dicionarios (já com nulos tratados).

    Returns:
        int: Quantidade de categorias efetivamente modificadas.
    """
    modificadas = 0

    for produto in produtos:
        bruta = produto.get("product_category_name")
        padronizada = padronizar_categoria(bruta)

        # Contabiliza apenas quando houve alteração real
        if padronizada != bruta:
            modificadas += 1
        produto["product_category_name"] = padronizada

    return modificadas


# =========================================================
# Funções de lógica de Regra de Negócio
# =========================================================

def classificar_pedido_sem_entrega(pedido):
    """
    Classifica o motivo de um pedido sem data de entrega usando if/elif/else.

    Args:
        pedido (dict): Pedido cujo order_delivered_customer_date está vazio.

    Returns:
        str: Classificação do pedido (motivo provável da data nula).
    """
    status = pedido.get("order_status", "").strip().lower()

    if status == "canceled":
        return "cancelado"
    elif status == "delivered":
        # Caso anômalo: status entregue, porém sem data registrada
        return "entregue_sem_data"
    elif status in ("shipped", "invoiced", "processing", "created", "approved"):
        return "em_andamento"
    elif status == "unavailable":
        return "indisponivel"
    else:
        return "status_desconhecido"


def separar_pedidos_sem_entrega(pedidos):
    """
    Separa os pedidos sem data de entrega e classifica cada um pelo status.

    Args:
        pedidos (list): Lista de dicionários com os pedidos brutos.

    Returns:
        tuple: (pedidos_sem_entrega, classificacao_contadores, total_cancelados)
    """
    pedidos_sem_entrega = []
    contadores = {"cancelado": 0, "entregue_sem_data": 0, "em_andamento": 0,
                  "indisponivel": 0, "status_desconhecido": 0}
    total_cancelados = 0

    for pedido in pedidos:
        # Se o pedido e cancelado, contabiliza separadamente (para o relatório)
        if pedido.get("order_status", "").strip().lower() == "canceled":
            total_cancelados += 1

        # Aplica a regra de negocio apenas a quem não possui data de entrega
        if testar_valor_ausente(pedido.get("order_delivered_customer_date")):
            pedidos_sem_entrega.append(pedido)
            classificacao = classificar_pedido_sem_entrega(pedido)
            contadores[classificacao] += 1

    return pedidos_sem_entrega, contadores, total_cancelados


# =========================================================
# Funções de formatação de dados temporais
# =========================================================

def formatar_data_aprovacao_br(pedido):
    """
    Converte order_approved_at de 'YYYY-MM-DD HH:MM:SS' para 'DD/MM/YYYY'.

    Args:
        pedido (dict): Registro individual do pedido.

    Returns:
        bool: True se a conversão foi realizada, False caso contrário.
    """
    bruta = pedido.get("order_approved_at")

    if testar_valor_ausente(bruta):
        # Mantém vazio: o tratamento consolidado de nulos é feito em outra etapa
        pedido["order_approved_at_br"] = ""
        return False

    try:
        # Converte a string ISO para objeto datetime
        data_obj = datetime.strptime(bruta.strip(), "%Y-%m-%d %H:%M:%S")
        # Reformatada para o padrão brasileiro dd/mm/aaaa
        pedido["order_approved_at_br"] = data_obj.strftime("%d/%m/%Y")
        return True
    except ValueError:
        # Formato inesperado: preserva o valor original e sinaliza não conversão
        pedido["order_approved_at_br"] = bruta
        return False


def aplicar_formatacao_datas(pedidos):
    """
    Aplica a formatação da data de aprovação em todos os pedidos.

    Args:
        pedidos (list): Lista de dicionários com os pedidos brutos.

    Returns:
        int: Quantidade de datas convertidas com sucesso.
    """
    convertidas = 0

    for pedido in pedidos:
        if formatar_data_aprovacao_br(pedido):
            convertidas += 1

    return convertidas



# =========================================================
# Tratamento do arquivo de pedidos
# =========================================================

def sanitizar_pedidos(caminho_arquivo):
    """
    Orquestra a lógica de tratamento do arquivo de pedidos.

    Args:
        caminho_arquivo (str): Caminho do arquivo .csv de pedidos.

    Returns:
        tuple: (pedidos, pedidos_sem_entrega, contadores_classificacao,
                total_cancelados, datas_convertidas)
    """
    pedidos = ler_arquivo_pedidos(caminho_arquivo)

    # Regra de negócio: separacao e validação da hipótese
    pedidos_sem_entrega, contadores, total_cancelados = separar_pedidos_sem_entrega(pedidos)

    # Formatação temporal (datetime)
    datas_convertidas = aplicar_formatacao_datas(pedidos)

    return pedidos, pedidos_sem_entrega, contadores, total_cancelados, datas_convertidas


def escrever_base_sanitizada(registros, cabecalhos_extra, caminho_saida):
    """
    Grava um dataset sanitizado em CSV, incluindo colunas extras
    criadas pelas etapas do pipeline (ex.: order_approved_at_br).

    Args:
        registros (list): Lista de dicionários já sanitizados.
        cabecalhos_extra (list): Colunas extras que devem ser gravadas.
        caminho_saida (str): Caminho do arquivo de saida.

    Returns:
        int: Quantidade de linhas gravadas.
    """
    if not registros:
        return 0

    cabecalhos = list(registros[0].keys()) + [c for c in cabecalhos_extra
                                              if c not in registros[0]]
    # Remove possíveis duplicatas preservando a ordem
    vistos = set()
    cabecalhos_unicos = []
    for c in cabecalhos:
        if c not in vistos:
            vistos.add(c)
            cabecalhos_unicos.append(c)

    with open(caminho_saida, mode="w", encoding="utf-8", newline="") as arquivo:
        escritor = csv.DictWriter(arquivo, fieldnames=cabecalhos_unicos,
                                  extrasaction="ignore")
        escritor.writeheader()
        for registro in registros:
            escritor.writerow(registro)

    return len(registros)


---
# Execução do Pipeline Completo

Executamos agora todas as etapas na ordem correta e consolidamos o relatório final.

In [8]:
# ----------------------------------------------------------------------
# Caminhos e preparação do diretório de trabalho
# ----------------------------------------------------------------------
print("Iniciando a leitura dos dados...\n")

CAMINHO_DADOS = "data"
CAMINHO_PRODUTOS = "data/raw/olist_products_dataset.csv"
CAMINHO_PEDIDOS = "data/raw/olist_orders_dataset.csv"

CAMINHO_PRODUTOS_OUT = os.path.join(CAMINHO_DADOS, "olist_products_sanitizado.csv")
CAMINHO_PEDIDOS_OUT = os.path.join(CAMINHO_DADOS, "olist_orders_sanitizado.csv")

os.makedirs(CAMINHO_DADOS, exist_ok=True)

print("Executando sanitização e padronização de dados...")

# ----------------------------------------------------------------------
# Tratamento de dados ausentes (produtos)
# ----------------------------------------------------------------------
resultado_prod = sanitizar_produtos(CAMINHO_PRODUTOS)
produtos, total_prod, cat_corrigidas, dim_corrigidas, medias = resultado_prod

# ----------------------------------------------------------------------
# Padronização de strings e regex (produtos)
# ----------------------------------------------------------------------
categorias_padronizadas = aplicar_padronizacao_categorias(produtos)

# ----------------------------------------------------------------------
# Regra de negócio e formatação temporal (pedidos)
# ----------------------------------------------------------------------
resultado_ped = sanitizar_pedidos(CAMINHO_PEDIDOS)
(pedidos, pedidos_sem_entrega, contadores, total_cancelados,
 datas_convertidas) = resultado_ped

# ----------------------------------------------------------------------
# Persistência das bases de dados tratadas
# ----------------------------------------------------------------------
print("\nExecução completa, salvando os arquivos sanitizados...")
linhas_prod = escrever_produtos_sanitizados(produtos, CAMINHO_PRODUTOS_OUT)
linhas_ped = escrever_base_sanitizada(pedidos, ["order_approved_at_br"],
                                      CAMINHO_PEDIDOS_OUT)

Iniciando a leitura dos dados...

Executando sanitização e padronização de dados...

Execução completa, salvando os arquivos sanitizados...


In [9]:
# ----------------------------------------------------------------------
# RELATÓRIO DE STATUS MANUAL - PIPELINE COMPLETO
# ----------------------------------------------------------------------
print("=" * 62)
print(" RELATÓRIO DE STATUS - SANITIZAÇÃO COMPLETA")
print("=" * 62)

print("\n  1. Dados ausentes (produtos)")
print(f"   Total de produtos processados       : {total_prod}")
print(f"   Categorias nulas corrigidas         : {cat_corrigidas}")
print(f"   Valores físicos imputados (média)   : {sum(dim_corrigidas.values())}")

print("\n 2. Padronização de strings (produtos)")
print(f"   Categorias padronizadas (strip/lower/regex): {categorias_padronizadas}")

print("\n 3. Regras de negócio (pedidos)")
print(f"   Total de pedidos processados        : {len(pedidos)}")
print(f"   Pedidos cancelados identificados    : {total_cancelados}")
print(f"   Pedidos sem data de entrega         : {len(pedidos_sem_entrega)}")
print(f"   Dentre os sem data de entrega:")
for chave, qtd in contadores.items():
    print(f"      - {chave:20s}: {qtd}")

# Validação da hipótese de negócio
hipotese_confirmada = (contadores["cancelado"] == len(pedidos_sem_entrega)
                       and len(pedidos_sem_entrega) > 0)
print(f"\n   HIPÓTESE (data nula => pedido cancelado): "
      f"{'CONFIRMADA' if hipotese_confirmada else 'REFUTADA'}")
if not hipotese_confirmada:
    print("   -> Existem pedidos não cancelados sem data de entrega; a hipótese")
    print("      de negócio NÃO se sustenta para toda a base.")

print("\n 4. Formatação temporal (pedidos)")
print(f"   Datas de aprovação convertidas (dd/mm/aaaa): {datas_convertidas}")

print("\n 5. Persistência e validação final")
print(f"   Produtos gravados (CSV sanitizado)  : {linhas_prod}")
print(f"   Pedidos gravados (CSV sanitizado)   : {linhas_ped}")
print("=" * 62)

# Validação de integridade da base de produtos (0 valores ausentes)
campos_validar = ["product_category_name"] + COLUNAS_DIMENSOES_FISICAS
pendentes = sum(
    1 for p in produtos for c in campos_validar if testar_valor_ausente(p.get(c))
)
print("   Produtos - valores ausentes restantes: " + ("0  -> OK" if pendentes == 0
                                                      else str(pendentes)))
print("   Pedidos - coluna extra order_approved_at_br criada: OK")
print("=" * 62)

 RELATÓRIO DE STATUS - SANITIZAÇÃO COMPLETA

  1. Dados ausentes (produtos)
   Total de produtos processados       : 32951
   Categorias nulas corrigidas         : 610
   Valores físicos imputados (média)   : 8

 2. Padronização de strings (produtos)
   Categorias padronizadas (strip/lower/regex): 0

 3. Regras de negócio (pedidos)
   Total de pedidos processados        : 99441
   Pedidos cancelados identificados    : 625
   Pedidos sem data de entrega         : 2965
   Dentre os sem data de entrega:
      - cancelado           : 619
      - entregue_sem_data   : 8
      - em_andamento        : 1729
      - indisponivel        : 609
      - status_desconhecido : 0

   HIPÓTESE (data nula => pedido cancelado): REFUTADA
   -> Existem pedidos não cancelados sem data de entrega; a hipótese
      de negócio NÃO se sustenta para toda a base.

 4. Formatação temporal (pedidos)
   Datas de aprovação convertidas (dd/mm/aaaa): 99281

 5. Persistência e validação final
   Produtos gravados (CSV s

In [10]:
# ----------------------------------------------------------------------
# VERIFICAÇÃO VISUAL DOS RESULTADOS
# ----------------------------------------------------------------------
print("1) Amostra de categorias padronizadas:")
for produto in produtos[:3]:
    print("   -", produto["product_category_name"])

print("\n2) Pedidos cancelados sem data de entrega:")
exibidos = 0
for pedido in pedidos_sem_entrega:
    if pedido["order_status"].strip().lower() == "canceled":
        print("   -", pedido["order_id"], "|", pedido["order_status"],
              "| entrega:", repr(pedido["order_delivered_customer_date"]))
        exibidos += 1
        if exibidos == 3:
            break

print("\n3) Datas de aprovação no formato brasileiro:")
for pedido in pedidos[:5]:
    print("   -", pedido["order_approved_at"], "->", pedido["order_approved_at_br"])

1) Amostra de categorias padronizadas:
   - perfumaria
   - artes
   - esporte_lazer

2) Pedidos cancelados sem data de entrega:
   - 1b9ecfe83cdc259250e1a8aca174f0ad | canceled | entrega: ''
   - 714fb133a6730ab81fa1d3c1b2007291 | canceled | entrega: ''
   - 3a129877493c8189c59c60eb71d97c29 | canceled | entrega: ''

3) Datas de aprovação no formato brasileiro:
   - 2017-10-02 11:07:15 -> 02/10/2017
   - 2018-07-26 03:24:27 -> 26/07/2018
   - 2018-08-08 08:55:23 -> 08/08/2018
   - 2017-11-18 19:45:59 -> 18/11/2017
   - 2018-02-13 22:20:29 -> 13/02/2018


---
# Conclusão

O pipeline de sanitização das bases da Olist foi implementado em 5 etapas
utilizando **apenas bibliotecas nativas** (`csv`, `re`, `datetime`), com código modular
(encapsulado em funções), estruturas condicionais completas (`if/elif/else`) e laços de
repetição para varredura linha a linha.

O tratamento correto dos dados evita o problema de *Garbage In, Garbage Out*: bases
limpas e padronizadas reduzem o risco de **overfitting** e **viés** em futuros modelos
de Inteligência Artificial.

**Principais resultados:**

| Métrica | Valor |
|---|---|
| Produtos processados | 32.951 |
| Categorias nulas corrigidas | 610 |
| Valores físicos imputados pela média | 8 |
| Categorias padronizadas | 0 (base já estava padronizada) |
| Pedidos processados | 99.441 |
| Pedidos cancelados identificados | 625 |
| Pedidos sem data de entrega separados | 2.965 |
| Datas de aprovação convertidas | 99.281 |

A **hipótese de negócio** (data de entrega nula implica pedido cancelado) foi
**refutada** pelos dados: apenas 619 dos 2.965 pedidos sem data de entrega são
cancelados; o restante se divide entre outros status (`shipped`, `unavailable`,
`invoiced`, `processing`, `delivered`, etc.).